
# Wairau Creek GRU — High-Water-Risk Classification

This notebook develops a **Wairau Creek-specific GRU** using the same core GRU configuration used for the other catchments.

### Objective

Use the previous **48 hours** of:

- `rainfall_mm`
- `discharge_m3s`
- `water_level_m`

to estimate whether Wairau Creek is in a **high-water-risk window for the following 24 hours**.

### Important target definition

The supplied `wairau_model_ready_data.csv` already contains the target `high_water_risk_24h`.

Before modelling, this notebook independently checks that:

- the training-only 99.5th percentile water-level threshold is approximately **0.929 m**;
- the supplied target is consistent with the statistical high-water-risk methodology.

The supplied target is retained for modelling so that the GRU uses the same prepared labels as the team's other classifiers.

### Leakage controls

- Training, validation, and test periods come from the supplied `data_split`.
- Scaling is fitted on training data only.
- Probability threshold selection uses validation only.
- Test data are used only for final evaluation.
- Missing timestamps are restored as empty hourly rows so an GRU sequence can never jump across a time gap.
- Only fully continuous 48-hour input windows are used.

The target is a **statistical high-water-risk proxy**, not an independently verified historical flood label.


## Fair Comparison with LSTM

This GRU experiment intentionally keeps the **same modelling protocol as the corresponding LSTM**:

- same catchment data and target;
- same chronological Training / Validation / Test periods;
- same three inputs: `rainfall_mm`, `discharge_m3s`, and `water_level_m`;
- same 48-hour lookback;
- same 24-hour high-water-risk prediction horizon;
- same training-only scaling;
- same class-imbalance weighting;
- same hidden size (32), dense layer (16), dropout (0.25), optimizer, learning rate, batch size, early stopping, and validation-based probability-threshold selection;
- same evaluation metrics and episode-level diagnostic.

The **recurrent unit is the main architectural change**: `nn.LSTM` is replaced by `nn.GRU`. This makes the GRU-vs-LSTM comparison substantially more controlled.

In [1]:

import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.14.1+cpu
CUDA available: False


## 2. Load the Wairau Model-Ready Dataset

In [2]:

data_path = Path("wairau_model_ready_data.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "wairau_model_ready_data.csv was not found. "
        "Place it in the same folder as this notebook."
    )

source_data = pd.read_csv(data_path)

required_columns = [
    "timestamp",
    "data_split",
    "rainfall_mm",
    "discharge_m3s",
    "water_level_m",
    "high_water_risk_24h",
]

missing_columns = [
    column
    for column in required_columns
    if column not in source_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

source_data["timestamp"] = pd.to_datetime(
    source_data["timestamp"],
    errors="coerce"
)

source_data = (
    source_data
    .sort_values("timestamp")
    .reset_index(drop=True)
)

display(source_data.head())
print("Source shape:", source_data.shape)


,timestamp,data_split,rainfall_mm,water_level_m,discharge_m3s,rainfall_sum_3h,rainfall_sum_6h,rainfall_sum_12h,rainfall_sum_24h,rainfall_sum_48h,water_level_lag_1h,water_level_lag_3h,water_level_lag_6h,water_level_lag_12h,water_level_lag_24h,water_level_lag_48h,discharge_lag_1h,discharge_lag_3h,discharge_lag_6h,discharge_lag_12h,discharge_lag_24h,discharge_lag_48h,water_level_change_1h,water_level_change_3h,water_level_change_24h,discharge_change_1h,discharge_change_3h,discharge_change_24h,hour_sin,hour_cos,month_sin,month_cos,high_water_risk_24h
0,2015-01-03 01:00:00,Training,0.0,0.101,0.040,0.0,0.0,0.0,0.0,5.16802,0.102,0.101,0.096,0.107,0.111,0.112,0.041,0.040,0.035,0.046,0.051,0.052,-0.001,0.000,-0.010,-0.001,0.000,-0.011,0.258819,0.965926,0.0,1.0,0
1,2015-01-03 02:00:00,Training,0.0,0.104,0.043,0.0,0.0,0.0,0.0,3.05000,0.101,0.101,0.103,0.109,0.111,0.574,0.040,0.040,0.042,0.049,0.051,3.763,0.003,0.003,-0.007,0.003,0.003,-0.008,0.500000,0.866025,0.0,1.0,0
2,2015-01-03 03:00:00,Training,0.0,0.100,0.039,0.0,0.0,0.0,0.0,0.00000,0.104,0.102,0.101,0.109,0.109,0.585,0.043,0.041,0.040,0.049,0.049,3.920,-0.004,-0.002,-0.009,-0.004,-0.002,-0.010,0.707107,0.707107,0.0,1.0,0
3,2015-01-03 04:00:00,Training,0.0,0.107,0.046,0.0,0.0,0.0,0.0,0.00000,0.100,0.101,0.101,0.104,0.109,0.581,0.039,0.040,0.040,0.043,0.049,3.863,0.007,0.006,-0.002,0.007,0.006,-0.003,0.866025,0.500000,0.0,1.0,0
4,2015-01-03 05:00:00,Training,0.0,0.107,0.046,0.0,0.0,0.0,0.0,0.00000,0.107,0.104,0.101,0.102,0.108,0.302,0.046,0.043,0.040,0.041,0.048,0.763,0.000,0.003,-0.001,0.000,0.003,-0.002,0.965926,0.258819,0.0,1.0,0


Source shape: (94643, 33)


## 3. Audit the Supplied Dataset

In [3]:

feature_columns = [
    "rainfall_mm",
    "discharge_m3s",
    "water_level_m",
]

print("Date range:")
print(
    source_data["timestamp"].min(),
    "to",
    source_data["timestamp"].max()
)

print("\nDuplicate timestamps:")
print(
    source_data["timestamp"].duplicated().sum()
)

print("\nMissing values in GRU inputs:")
print(
    source_data[
        feature_columns
    ].isna().sum()
)

print("\nRows by supplied data split:")
print(
    source_data[
        "data_split"
    ].value_counts()
)

time_difference = (
    source_data["timestamp"].diff()
)

gap_mask = (
    time_difference.notna()
    & (
        time_difference
        != pd.Timedelta(hours=1)
    )
)

print("\nNon-hourly timestamp gaps:")
print(
    gap_mask.sum()
)

if gap_mask.sum() > 0:
    gap_audit = pd.DataFrame({
        "gap_end": source_data.loc[
            gap_mask,
            "timestamp"
        ].values,
        "gap_size": time_difference.loc[
            gap_mask
        ].values,
    })

    display(
        gap_audit.head(20)
    )


Date range:
2015-01-03 01:00:00 to 2025-12-31 00:00:00

Duplicate timestamps:
0

Missing values in GRU inputs:
rainfall_mm      0
discharge_m3s    0
water_level_m    0
dtype: int64

Rows by supplied data split:
data_split
Training      65944
Test          14427
Validation    14272
Name: count, dtype: int64

Non-hourly timestamp gaps:
80


,gap_end,gap_size
0,2015-03-11 17:00:00,0 days 10:00:00
1,2015-03-11 23:00:00,0 days 04:00:00
2,2015-03-12 11:00:00,0 days 04:00:00
3,2015-03-13 11:00:00,0 days 04:00:00
4,2016-03-23 19:00:00,0 days 06:00:00
5,2016-03-23 22:00:00,0 days 03:00:00
6,2016-03-24 04:00:00,0 days 03:00:00
7,2016-03-24 16:00:00,0 days 03:00:00
8,2016-03-25 16:00:00,0 days 03:00:00
9,2016-03-30 15:00:00,0 days 06:00:00



## 4. Restore the Complete Hourly Timeline

The supplied model-ready dataset contains intentional and data-quality-related gaps.

For Random Forest or XGBoost this is not necessarily a problem because each row can be evaluated independently.

For an GRU, however, a 48-hour sequence must represent **48 consecutive hours**. Therefore, the full hourly timeline is restored and absent timestamps remain `NaN`.

No missing measurement is invented or interpolated.


In [4]:

hourly_timeline = pd.DataFrame({
    "timestamp": pd.date_range(
        start=source_data["timestamp"].min(),
        end=source_data["timestamp"].max(),
        freq="h"
    )
})

model_data = hourly_timeline.merge(
    source_data[
        [
            "timestamp",
            "data_split",
            "rainfall_mm",
            "discharge_m3s",
            "water_level_m",
            "high_water_risk_24h",
        ]
    ],
    on="timestamp",
    how="left"
)

print("Restored hourly shape:", model_data.shape)

print(
    "Rows restored as missing timestamps:",
    model_data["data_split"].isna().sum()
)

display(model_data.head())


Restored hourly shape: (96360, 6)
Rows restored as missing timestamps: 1717


,timestamp,data_split,rainfall_mm,discharge_m3s,water_level_m,high_water_risk_24h
0,2015-01-03 01:00:00,Training,0.0,0.040,0.101,0.0
1,2015-01-03 02:00:00,Training,0.0,0.043,0.104,0.0
2,2015-01-03 03:00:00,Training,0.0,0.039,0.100,0.0
3,2015-01-03 04:00:00,Training,0.0,0.046,0.107,0.0
4,2015-01-03 05:00:00,Training,0.0,0.046,0.107,0.0


## 5. Verify the Statistical High-Water Threshold

In [5]:

HIGH_WATER_PERCENTILE = 0.995

training_water_levels = (
    model_data.loc[
        model_data["data_split"].eq("Training"),
        "water_level_m"
    ]
    .dropna()
)

high_water_threshold = (
    training_water_levels.quantile(
        HIGH_WATER_PERCENTILE
    )
)

print(
    "Training observations used for threshold:",
    len(training_water_levels)
)

print(
    "Training-only 99.5th percentile high-water threshold:",
    round(
        high_water_threshold,
        3
    ),
    "m"
)


Training observations used for threshold: 65944
Training-only 99.5th percentile high-water threshold: 0.929 m



### Independent target consistency check

The next cell independently rebuilds a 24-hour high-water-risk target wherever all 24 future hourly water-level readings are available.

This rebuilt label is used **only as a consistency check**. The supplied `high_water_risk_24h` remains the modelling target.


In [6]:

FORECAST_HORIZON = 24

future_water_levels = pd.concat(
    [
        model_data["water_level_m"].shift(-hour)
        for hour in range(
            1,
            FORECAST_HORIZON + 1
        )
    ],
    axis=1
)

independent_future_max = (
    future_water_levels.max(
        axis=1,
        skipna=False
    )
)

independent_target = np.where(
    independent_future_max.isna(),
    np.nan,
    (
        independent_future_max
        >= high_water_threshold
    ).astype(int)
)

target_check = pd.DataFrame({
    "supplied_target":
        model_data[
            "high_water_risk_24h"
        ],
    "independent_target":
        independent_target,
}).dropna()

agreement_rate = (
    target_check[
        "supplied_target"
    ].astype(int)
    ==
    target_check[
        "independent_target"
    ].astype(int)
).mean()

print(
    "Comparable rows:",
    len(target_check)
)

print(
    "Agreement between supplied and independently reconstructed target:",
    f"{agreement_rate * 100:.3f}%"
)

print(
    "Number of differing comparable rows:",
    int(
        (
            target_check[
                "supplied_target"
            ].astype(int)
            !=
            target_check[
                "independent_target"
            ].astype(int)
        ).sum()
    )
)


Comparable rows: 93635
Agreement between supplied and independently reconstructed target: 99.924%
Number of differing comparable rows: 71


## 6. Review the Supplied Chronological Splits

In [7]:

split_summary = (
    model_data
    .dropna(
        subset=["data_split"]
    )
    .groupby("data_split")
    .agg(
        rows=("timestamp", "size"),
        beginning=("timestamp", "min"),
        ending=("timestamp", "max"),
        positive_hours=(
            "high_water_risk_24h",
            "sum"
        ),
    )
)

split_summary[
    "positive_percentage"
] = (
    100
    * split_summary[
        "positive_hours"
    ]
    / split_summary[
        "rows"
    ]
)

display(
    split_summary.loc[
        [
            "Training",
            "Validation",
            "Test"
        ]
    ].round(3)
)


,rows,beginning,ending,positive_hours,positive_percentage
data_split,,,,,
Training,65944,2015-01-03 01:00:00,2022-09-12 14:00:00,3361.0,5.097
Validation,14272,2022-09-13 15:00:00,2024-05-07 07:00:00,890.0,6.236
Test,14427,2024-05-08 08:00:00,2025-12-31 00:00:00,863.0,5.982


## 7. Identify Valid Continuous Input Hours

In [8]:

model_data["features_available"] = (
    model_data[
        feature_columns
    ]
    .notna()
    .all(axis=1)
)

missing_feature_rows = (
    ~model_data[
        "features_available"
    ]
)

print(
    "Hourly rows with at least one unavailable input:",
    missing_feature_rows.sum()
)

missing_groups = (
    missing_feature_rows
    .ne(
        missing_feature_rows.shift(
            fill_value=False
        )
    )
    .cumsum()
)

missing_episodes = (
    model_data.loc[
        missing_feature_rows
    ]
    .assign(
        missing_group=missing_groups[
            missing_feature_rows
        ]
    )
    .groupby("missing_group")
    .agg(
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        hours=("timestamp", "size"),
    )
    .reset_index(drop=True)
)

print(
    "Missing-data / restored-gap episodes:",
    len(missing_episodes)
)

if len(missing_episodes) > 0:
    print(
        "Longest unavailable-input episode:",
        missing_episodes[
            "hours"
        ].max(),
        "hours"
    )

    display(
        missing_episodes.head(20)
    )


Hourly rows with at least one unavailable input: 1717
Missing-data / restored-gap episodes: 80
Longest unavailable-input episode: 646 hours


,start,end,hours
0,2015-03-11 08:00:00,2015-03-11 16:00:00,9
1,2015-03-11 20:00:00,2015-03-11 22:00:00,3
2,2015-03-12 08:00:00,2015-03-12 10:00:00,3
3,2015-03-13 08:00:00,2015-03-13 10:00:00,3
4,2016-03-23 14:00:00,2016-03-23 18:00:00,5
5,2016-03-23 20:00:00,2016-03-23 21:00:00,2
6,2016-03-24 02:00:00,2016-03-24 03:00:00,2
7,2016-03-24 14:00:00,2016-03-24 15:00:00,2
8,2016-03-25 14:00:00,2016-03-25 15:00:00,2
9,2016-03-30 10:00:00,2016-03-30 14:00:00,5


## 8. GRU Lookback

In [9]:

LOOKBACK_HOURS = 48

candidate_lookbacks = [
    24,
    48,
    72,
]

candidate_results = []

for lookback in candidate_lookbacks:

    complete_input_window = (
        model_data[
            "features_available"
        ]
        .astype(int)
        .rolling(
            window=lookback,
            min_periods=lookback
        )
        .sum()
        .eq(lookback)
    )

    for period_name in [
        "Training",
        "Validation",
        "Test"
    ]:

        same_period_window = (
            model_data[
                "data_split"
            ]
            .eq(period_name)
            .astype(int)
            .rolling(
                window=lookback,
                min_periods=lookback
            )
            .sum()
            .eq(lookback)
        )

        valid_samples = (
            complete_input_window
            & same_period_window
            & model_data[
                "high_water_risk_24h"
            ].notna()
        )

        candidate_results.append({
            "lookback_hours":
                lookback,
            "data_split":
                period_name,
            "valid_samples":
                int(
                    valid_samples.sum()
                ),
        })

lookback_summary = pd.DataFrame(
    candidate_results
)

display(
    lookback_summary.pivot(
        index="lookback_hours",
        columns="data_split",
        values="valid_samples"
    )
)


data_split,Test,Training,Validation
lookback_hours,,,
24,14274,65264,14117
48,14202,64952,14021
72,14130,64640,13925


## 9. Scale the Three GRU Inputs Using Training Data Only

In [10]:

scaler_fit_mask = (
    model_data[
        "data_split"
    ].eq("Training")
    & model_data[
        "features_available"
    ]
)

scaler = StandardScaler()

scaler.fit(
    model_data.loc[
        scaler_fit_mask,
        feature_columns
    ]
)

scaled_features = np.full(
    (
        len(model_data),
        len(feature_columns)
    ),
    np.nan,
    dtype=np.float32
)

available_input_mask = (
    model_data[
        "features_available"
    ]
)

scaled_features[
    available_input_mask.to_numpy()
] = scaler.transform(
    model_data.loc[
        available_input_mask,
        feature_columns
    ]
).astype(np.float32)

print("Training means:")
display(
    pd.Series(
        scaler.mean_,
        index=feature_columns
    ).to_frame("mean")
)

print("\nTraining scales:")
display(
    pd.Series(
        scaler.scale_,
        index=feature_columns
    ).to_frame("scale")
)


Training means:


,mean
rainfall_mm,0.138351
discharge_m3s,0.370538
water_level_m,0.159540



Training scales:


,scale
rainfall_mm,0.781617
discharge_m3s,1.661808
water_level_m,0.132486


## 10. Convert the Hourly Series into GRU Sequences

In [11]:

from numpy.lib.stride_tricks import sliding_window_view

def create_sequences(
    data,
    scaled_feature_array,
    lookback,
    period_name,
):
    targets = (
        data[
            "high_water_risk_24h"
        ]
        .to_numpy()
    )

    periods = (
        data[
            "data_split"
        ]
        .to_numpy()
    )

    feature_available = (
        np.isfinite(
            scaled_feature_array
        )
        .all(axis=1)
    )

    complete_feature_window = (
        pd.Series(
            feature_available.astype(int)
        )
        .rolling(
            window=lookback,
            min_periods=lookback
        )
        .sum()
        .to_numpy()
        == lookback
    )

    same_period_window = (
        pd.Series(
            (
                periods
                == period_name
            ).astype(int)
        )
        .rolling(
            window=lookback,
            min_periods=lookback
        )
        .sum()
        .to_numpy()
        == lookback
    )

    valid_end = (
        complete_feature_window
        & same_period_window
        & np.isfinite(targets)
    )

    end_indices = np.where(
        valid_end
    )[0]

    all_windows = (
        sliding_window_view(
            scaled_feature_array,
            window_shape=lookback,
            axis=0
        )
        .transpose(
            0,
            2,
            1
        )
    )

    window_positions = (
        end_indices
        - (lookback - 1)
    )

    X = (
        all_windows[
            window_positions
        ]
        .copy()
        .astype(np.float32)
    )

    y = (
        targets[
            end_indices
        ]
        .astype(np.float32)
    )

    timestamps = (
        data.loc[
            end_indices,
            "timestamp"
        ]
        .to_numpy()
    )

    return (
        X,
        y,
        timestamps,
        end_indices,
    )


In [12]:

X_train, y_train, train_timestamps, train_indices = (
    create_sequences(
        model_data,
        scaled_features,
        LOOKBACK_HOURS,
        "Training"
    )
)

X_validation, y_validation, validation_timestamps, validation_indices = (
    create_sequences(
        model_data,
        scaled_features,
        LOOKBACK_HOURS,
        "Validation"
    )
)

print(
    "X_train:",
    X_train.shape
)

print(
    "X_validation:",
    X_validation.shape
)

print("\nPositive class percentage:")
print(
    "Training:",
    round(
        y_train.mean() * 100,
        3
    ),
    "%"
)

print(
    "Validation:",
    round(
        y_validation.mean() * 100,
        3
    ),
    "%"
)


X_train: (64952, 48, 3)
X_validation: (14021, 48, 3)

Positive class percentage:
Training: 5.145 %
Validation: 6.348 %


## 11. Handle Class Imbalance

In [13]:

negative_count = int(
    (y_train == 0).sum()
)

positive_count = int(
    (y_train == 1).sum()
)

positive_weight = (
    negative_count
    / positive_count
)

print(
    "Negative training samples:",
    negative_count
)

print(
    "Positive training samples:",
    positive_count
)

print(
    "Positive class weight:",
    round(
        positive_weight,
        3
    )
)


Negative training samples: 61610
Positive training samples: 3342
Positive class weight: 18.435


## 12. GRU Architecture

In [14]:

class GRUClassifier(
    nn.Module
):
    def __init__(
        self,
        input_size=3,
        hidden_size=32,
        dense_size=16,
        dropout=0.25,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.dense = nn.Linear(
            hidden_size,
            dense_size
        )

        self.relu = nn.ReLU()

        self.output = nn.Linear(
            dense_size,
            1
        )

    def forward(
        self,
        x
    ):
        _, hidden_state = self.gru(x)

        x = hidden_state[-1]
        x = self.dropout(x)
        x = self.dense(x)
        x = self.relu(x)

        logits = self.output(x)

        return logits.squeeze(1)


model = GRUClassifier(
    input_size=len(
        feature_columns
    ),
    hidden_size=32,
    dense_size=16,
    dropout=0.25,
)

print(model)


GRUClassifier(
  (gru): GRU(3, 32, batch_first=True)
  (dropout): Dropout(p=0.25, inplace=False)
  (dense): Linear(in_features=32, out_features=16, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=16, out_features=1, bias=True)
)


## 13. Train the GRU

In [ ]:

torch.manual_seed(SEED)
np.random.seed(SEED)

BATCH_SIZE = 512
MAX_EPOCHS = 15
PATIENCE = 4
LEARNING_RATE = 0.001
MIN_DELTA = 1e-4

model = GRUClassifier(
    input_size=len(
        feature_columns
    ),
    hidden_size=32,
    dense_size=16,
    dropout=0.25,
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

loss_function = (
    nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            positive_weight,
            dtype=torch.float32
        )
    )
)

train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(
            X_train
        ),
        torch.from_numpy(
            y_train
        )
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

validation_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(
            X_validation
        ),
        torch.from_numpy(
            y_validation
        )
    ),
    batch_size=1024,
    shuffle=False,
)

best_validation_pr_auc = -np.inf
best_state = None
best_epoch = None
epochs_without_improvement = 0
training_history = []

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    model.train()

    total_loss = 0.0

    for (
        X_batch,
        y_batch
    ) in train_loader:

        optimizer.zero_grad()

        logits = model(
            X_batch
        )

        loss = loss_function(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += (
            loss.item()
            * len(y_batch)
        )

    epoch_loss = (
        total_loss
        / len(y_train)
    )

    model.eval()

    probability_parts = []

    with torch.no_grad():

        for (
            X_batch,
            _
        ) in validation_loader:

            probabilities = torch.sigmoid(
                model(
                    X_batch
                )
            )

            probability_parts.append(
                probabilities.cpu().numpy()
            )

    validation_probabilities = np.concatenate(
        probability_parts
    )

    validation_pr_auc = (
        average_precision_score(
            y_validation,
            validation_probabilities
        )
    )

    validation_roc_auc = (
        roc_auc_score(
            y_validation,
            validation_probabilities
        )
    )

    training_history.append({
        "epoch":
            epoch,
        "train_loss":
            epoch_loss,
        "validation_pr_auc":
            validation_pr_auc,
        "validation_roc_auc":
            validation_roc_auc,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"loss={epoch_loss:.4f} | "
        f"val_PR_AUC={validation_pr_auc:.4f} | "
        f"val_ROC_AUC={validation_roc_auc:.4f}"
    )

    if (
        validation_pr_auc
        >
        best_validation_pr_auc
        + MIN_DELTA
    ):

        best_validation_pr_auc = (
            validation_pr_auc
        )

        best_state = copy.deepcopy(
            model.state_dict()
        )

        best_epoch = epoch

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= PATIENCE
        ):
            print(
                "\nEarly stopping triggered."
            )
            break

model.load_state_dict(
    best_state
)

history_df = pd.DataFrame(
    training_history
)

print(
    "\nBest epoch:",
    best_epoch
)

print(
    "Best validation PR-AUC:",
    round(
        best_validation_pr_auc,
        4
    )
)


Epoch 01 | loss=1.2411 | val_PR_AUC=0.2978 | val_ROC_AUC=0.7510
Epoch 02 | loss=1.1669 | val_PR_AUC=0.3103 | val_ROC_AUC=0.7518
Epoch 03 | loss=1.1560 | val_PR_AUC=0.3156 | val_ROC_AUC=0.7522
Epoch 04 | loss=1.1525 | val_PR_AUC=0.3180 | val_ROC_AUC=0.7525
Epoch 05 | loss=1.1503 | val_PR_AUC=0.3188 | val_ROC_AUC=0.7527
Epoch 06 | loss=1.1479 | val_PR_AUC=0.3191 | val_ROC_AUC=0.7530


## 14. Select the Probability Threshold Using Validation Only

In [ ]:

model.eval()

with torch.no_grad():

    validation_probabilities = torch.sigmoid(
        model(
            torch.from_numpy(
                X_validation
            )
        )
    ).cpu().numpy()

(
    precision_curve,
    recall_curve,
    threshold_curve,
) = precision_recall_curve(
    y_validation,
    validation_probabilities
)

precision_for_threshold = (
    precision_curve[:-1]
)

recall_for_threshold = (
    recall_curve[:-1]
)

f2_curve = (
    5
    * precision_for_threshold
    * recall_for_threshold
    / (
        4
        * precision_for_threshold
        + recall_for_threshold
        + 1e-12
    )
)

best_threshold_position = (
    np.argmax(
        f2_curve
    )
)

selected_probability_threshold = float(
    threshold_curve[
        best_threshold_position
    ]
)

validation_predictions = (
    validation_probabilities
    >= selected_probability_threshold
).astype(int)

tn_val, fp_val, fn_val, tp_val = (
    confusion_matrix(
        y_validation,
        validation_predictions
    )
    .ravel()
)

validation_metrics = pd.Series({
    "threshold":
        selected_probability_threshold,
    "accuracy":
        accuracy_score(
            y_validation,
            validation_predictions
        ),
    "precision":
        precision_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
    "recall":
        recall_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
    "f1_score":
        f1_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
    "f2_score":
        fbeta_score(
            y_validation,
            validation_predictions,
            beta=2,
            zero_division=0
        ),
    "roc_auc":
        roc_auc_score(
            y_validation,
            validation_probabilities
        ),
    "pr_auc":
        average_precision_score(
            y_validation,
            validation_probabilities
        ),
    "true_positives":
        tp_val,
    "false_negatives":
        fn_val,
    "false_positives":
        fp_val,
    "true_negatives":
        tn_val,
})

display(
    validation_metrics
    .to_frame(
        "Validation"
    )
    .round(4)
)


## 15. Final Evaluation on the Untouched Test Set

In [ ]:

X_test, y_test, test_timestamps, test_indices = (
    create_sequences(
        model_data,
        scaled_features,
        LOOKBACK_HOURS,
        "Test"
    )
)

print(
    "X_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)


In [ ]:

model.eval()

test_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(
            X_test
        ),
        torch.from_numpy(
            y_test
        )
    ),
    batch_size=1024,
    shuffle=False,
)

test_probability_parts = []

with torch.no_grad():

    for (
        X_batch,
        _
    ) in test_loader:

        probabilities = torch.sigmoid(
            model(
                X_batch
            )
        )

        test_probability_parts.append(
            probabilities.cpu().numpy()
        )

test_probabilities = np.concatenate(
    test_probability_parts
)

test_predictions_binary = (
    test_probabilities
    >= selected_probability_threshold
).astype(int)

tn, fp, fn, tp = (
    confusion_matrix(
        y_test,
        test_predictions_binary
    )
    .ravel()
)

test_metrics = pd.Series({
    "selected_threshold":
        selected_probability_threshold,
    "accuracy":
        accuracy_score(
            y_test,
            test_predictions_binary
        ),
    "precision":
        precision_score(
            y_test,
            test_predictions_binary,
            zero_division=0
        ),
    "recall":
        recall_score(
            y_test,
            test_predictions_binary,
            zero_division=0
        ),
    "f1_score":
        f1_score(
            y_test,
            test_predictions_binary,
            zero_division=0
        ),
    "f2_score":
        fbeta_score(
            y_test,
            test_predictions_binary,
            beta=2,
            zero_division=0
        ),
    "roc_auc":
        roc_auc_score(
            y_test,
            test_probabilities
        ),
    "pr_auc":
        average_precision_score(
            y_test,
            test_probabilities
        ),
    "true_positives":
        tp,
    "false_negatives":
        fn,
    "false_positives":
        fp,
    "true_negatives":
        tn,
})

display(
    test_metrics
    .to_frame(
        "Test"
    )
    .round(4)
)

print("\nClassification report:")

print(
    classification_report(
        y_test,
        test_predictions_binary,
        digits=4,
        zero_division=0
    )
)


## 16. Test Diagnostic Plots

In [ ]:

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions_binary,
    display_labels=[
        "No high-water risk",
        "High-water risk"
    ],
    values_format="d"
)

plt.title(
    "Wairau GRU - Test Confusion Matrix"
)
plt.show()

RocCurveDisplay.from_predictions(
    y_test,
    test_probabilities
)

plt.title(
    "Wairau GRU - Test ROC Curve"
)
plt.show()

PrecisionRecallDisplay.from_predictions(
    y_test,
    test_probabilities
)

plt.title(
    "Wairau GRU - Test Precision-Recall Curve"
)
plt.show()


## 17. Hourly Test Predictions

In [ ]:

test_predictions = pd.DataFrame({
    "timestamp":
        pd.to_datetime(
            test_timestamps
        ),
    "actual_risk":
        y_test.astype(int),
    "risk_probability":
        test_probabilities,
    "predicted_risk":
        test_predictions_binary.astype(int),
})

display(
    test_predictions.head()
)

print(
    "Prediction rows:",
    len(test_predictions)
)



## 18. Episode-Level Diagnostic Evaluation

Consecutive positive target hours are grouped into high-water-risk episodes.

An episode is counted as detected if the GRU produces at least one positive prediction during that episode.

This is a diagnostic measure of **statistical high-water-risk episode detection**, not verification of officially recorded historical floods.


In [ ]:

full_test_target = (
    model_data.loc[
        model_data[
            "data_split"
        ].eq("Test")
        & model_data[
            "high_water_risk_24h"
        ].notna(),
        [
            "timestamp",
            "high_water_risk_24h",
        ]
    ]
    .rename(
        columns={
            "high_water_risk_24h":
                "actual_risk"
        }
    )
    .copy()
)

full_test_target[
    "actual_risk"
] = (
    full_test_target[
        "actual_risk"
    ]
    .astype(int)
)

event_frame = (
    full_test_target
    .merge(
        test_predictions[
            [
                "timestamp",
                "risk_probability",
                "predicted_risk",
            ]
        ],
        on="timestamp",
        how="left"
    )
    .sort_values(
        "timestamp"
    )
    .reset_index(
        drop=True
    )
)

event_frame[
    "predicted_risk_for_detection"
] = (
    event_frame[
        "predicted_risk"
    ]
    .fillna(0)
    .astype(int)
)

actual_positive = (
    event_frame[
        "actual_risk"
    ]
    == 1
)

timestamp_gap = (
    event_frame[
        "timestamp"
    ]
    .diff()
    .ne(
        pd.Timedelta(
            hours=1
        )
    )
)

new_event = (
    actual_positive
    & (
        ~actual_positive.shift(
            fill_value=False
        )
        | timestamp_gap
    )
)

event_frame[
    "event_id"
] = (
    new_event
    .cumsum()
    .where(
        actual_positive
    )
)

event_level_results = (
    event_frame.loc[
        actual_positive
    ]
    .groupby(
        "event_id"
    )
    .agg(
        event_start=(
            "timestamp",
            "min"
        ),
        event_end=(
            "timestamp",
            "max"
        ),
        event_duration_hours=(
            "timestamp",
            "size"
        ),
        detected_hours=(
            "predicted_risk_for_detection",
            "sum"
        ),
        maximum_risk_score=(
            "risk_probability",
            "max"
        ),
        mean_risk_score=(
            "risk_probability",
            "mean"
        ),
    )
    .reset_index()
)

event_level_results[
    "event_detected"
] = (
    event_level_results[
        "detected_hours"
    ]
    > 0
)

event_level_results[
    "hourly_detection_percentage"
] = (
    100
    * event_level_results[
        "detected_hours"
    ]
    / event_level_results[
        "event_duration_hours"
    ]
)

display(
    event_level_results
)

event_detection_rate = (
    event_level_results[
        "event_detected"
    ]
    .mean()
    * 100
)

print(
    "\nEvent detection rate:",
    round(
        event_detection_rate,
        2
    ),
    "%"
)


## 19. Save GRU Outputs

In [ ]:

output_dir = (
    Path("outputs")
    / "wairau_gru"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

torch.save(
    {
        "model_state_dict":
            model.state_dict(),
        "lookback_hours":
            LOOKBACK_HOURS,
        "feature_columns":
            feature_columns,
        "selected_probability_threshold":
            selected_probability_threshold,
        "high_water_threshold_m":
            float(
                high_water_threshold
            ),
    },
    output_dir
    / "wairau_gru_model.pt"
)

joblib.dump(
    scaler,
    output_dir
    / "wairau_gru_scaler.pkl"
)

test_predictions.to_csv(
    output_dir
    / "wairau_gru_test_predictions.csv",
    index=False
)

event_level_results.to_csv(
    output_dir
    / "wairau_gru_event_level_results.csv",
    index=False
)

validation_metrics.to_frame(
    "value"
).to_csv(
    output_dir
    / "wairau_gru_validation_metrics.csv"
)

test_metrics.to_frame(
    "value"
).to_csv(
    output_dir
    / "wairau_gru_test_metrics.csv"
)

history_df.to_csv(
    output_dir
    / "wairau_gru_training_history.csv",
    index=False
)

print(
    "Saved outputs to:",
    output_dir.resolve()
)



## 20. Interpretation

This GRU answers:

> **Given the previous 48 hours of rainfall, discharge, and water level in Wairau Creek, is the catchment in a statistical high-water-risk window for the following 24 hours?**

The input file already contains all three core GRU predictors, so **no additional consolidated hourly CSV is exported**.

The target should be described as **high-water risk**, not as an independently verified historical flood occurrence.


## GRU vs LSTM Interpretation Reminder

Do not decide whether GRU is better from accuracy alone. The main comparison should use **Validation F2** for model selection, with Recall, Precision, PR-AUC, ROC-AUC, and Test generalisation used as supporting evidence.

A higher GRU score does not automatically mean GRU is universally better; it means it performed better for that catchment under the same modelling protocol.